# 17 — Multi-timeframe pattern viewer

One chart, patterns from several timeframes. Watch BTC on 15m and see the 1h and 4h shapes and
trendlines that the 15m frame cannot know about — or drop to 5m/1m detail inside it.

Each extra timeframe is **detected in its own bar space** and placed here by timestamp, the only
coordinate the frames share. A 4h slope stays a 4h slope; nothing is rescaled or remapped.

### Three things to know before you read anything off it

**1. A higher-timeframe event is drawn where that bar CLOSED.** Bars carry their *open* time
everywhere in this project. A 4h pattern confirming on the bar stamped 12:00 is not knowable until
**16:00**, and that is where its ★ goes. The shape's own geometry — its boundaries and the pivots
they are fitted through — stays where it happened. Drawing the star at 12:00 instead would be a
four-hour lookahead on a 15m chart.

**2. Weight means timeframe, colour still means direction.**

| | drawn as |
|---|---|
| coarser than the chart | heavier, solid, fully opaque — this is *structure* |
| the chart's own timeframe | the normal 2.6px |
| finer than the chart | thinner, semi-transparent — this is *detail* |

Green is bullish, red bearish, amber neutral, exactly as before. Only the chart's own timeframe is
visible when it opens; everything else waits in the legend.

**3. Coarse timeframes need a WIDE window; fine ones need a NARROW one.** A 4h shape spans 100–200
4h bars — that is 17–33 days for the shape alone. Measured over a nine-day window, **4h confirmed
nothing at all**; that is arithmetic, not a bug. At the other end 1m confirmed **155** shapes over
the same window, which is the clutter rather than a feature.

| set | detection | shapes in view | build |
|---|---|---|---|
| `("4h","1h")` | 0.3 s | 3 | **5.5 s** |
| `+ "5m"` | 1.2 s | 26 | 6.9 s |
| `+ "1m"` | 6.1 s | 181 | 16.7 s |

### And the honest caveat

This is a **drawing** feature. Whether a 4h shape confirming under a 1h trend is worth trading has
**not been measured** — no phase-12 screen, no null differencing, no metronome comparison. The
research so far says these patterns predict *size*, not *direction*: hit rate **0.48**, below a coin
flip. See [`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md). Use this to look, not to conclude.


## 1. Config — edit here

In [ ]:
# the kernel starts in notebooks/, so put the repo root on the path (same as the other notebooks)
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import pandas as pd

SYMBOL = "BTCUSDT"
SOURCE = "binance"                     # or "dukascopy" with SYMBOL="XAUUSD"

TF          = "15m"                    # the CHART timeframe — the candles you actually see
PATTERN_TFS = ("4h", "1h")             # ALSO draw patterns from these. () = single-timeframe chart

VIEW_START, END = "2026-06-01", "2026-06-10"      # the window the chart opens on
WARMUP_START    = "2026-04-01"         # chart-timeframe data starts here and feeds the indicators

# Extra timeframes get their history in THEIR OWN bars, not in calendar days: a fixed start date
# starves 4h and drowns 1m. 800 bars is 33 days of 4h and 8 hours of 1m.
TF_WARMUP_BARS = 800

# None = all twenty-two patterns. Name a subset to declutter, e.g. ["rising_wedge", "double_top"].
PATTERNS = None

# ON when the chart opens; everything else starts hidden. "sr_zones" toggles the zones.
# "-ema50" turns one EMA off. Extra timeframes are named "1h::rising_wedge".
START_VISIBLE = ("sr_zones",)

SHOW_SR_ZONES   = True
SHOW_TRENDLINES = True                 # fitted support/resistance lines + their break markers
TF_TRENDLINES   = None                 # None = follow SHOW_TRENDLINES for the extra timeframes too
EMAS            = (20, 50, 100, 200)   # EMAs of the CHART timeframe

# EMAs of OTHER timeframes. Each entry is a TIMEFRAME AND ITS PERIODS — unlike PATTERN_TFS above,
# which is timeframes on their own. All four of these are accepted:
#     ("1h", 200)   ("1h", [50, 200])   ("1h", 50, 100, 200)   [("1h", 200), ("4h", 200)]
# Each line is a STEP that changes only at its own bar's close — the only moment the value is
# final. Hue is the timeframe and weight is the period, so 50/100/200 on two timeframes reads as
# two groups of three rather than six unrelated colours.
# A timeframe listed in PATTERN_TFS is reused from FRAMES (deeper history); anything else is
# resampled from the chart, which may not carry enough bars for a long period. You are warned when
# it does not, rather than left with an invisible line.
MTF_EMAS = [("1h", 50, 100, 200), ("4h", 200)]     # None for chart-timeframe EMAs only
LINE_PICKER     = True                 # the three cascading dropdowns (timeframe/pattern/occurrence)

# PIN families here to build a selection ACROSS timeframes — one 15m resistance line PLUS two 5m
# ones PLUS a few 1m shapes. Every occurrence of a pinned family becomes its own legend entry,
# starting off, and legend entries accumulate; the dropdowns cannot, because a plotly button is
# stateless and each press replaces the selection rather than adding to it.
#   "resistance_trendline"        the CHART timeframe's, like PATTERNS and START_VISIBLE
#   "5m::resistance_trendline"    that one timeframe's
#   "*::channel_up"               that family on every timeframe
#   "1m"                          every family of that timeframe
#   "all"                         everything (a legend hundreds of rows long — for a quick look)
# Spaces and underscores are the same thing. Cell 3 below prints every available key.
LEGEND_INSTANCES = None
ZONE_LR, CLUSTER_PCT, MIN_TOUCHES = 10, 0.15, 2
EXTEND, HEIGHT  = 12, 900
ZONE_LOOKBACK   = 2000                 # S/R zones are built from this many bars back from VIEW_START

# Clutter budget. Dropdown buttons carry a visibility vector as long as the trace list, so the menu
# costs buttons x traces — an uncapped 1m layer took the figure to 9.8 MB and it would not open.
MAX_PICKER    = 60                     # dropdown entries the EXTRA timeframes may add between them
MAX_SHAPES_TF = 40                     # shapes (and trendlines) drawn per pattern per timeframe

# standalone trendlines — the config reference at the bottom says what each of these does
TRENDLINE = dict(atr_mult=3.5, min_touches=3, lookback_pivots=16,
                 line_tol_atr=0.6, min_span_bars=40, max_age_bars=500,
                 break_buffer_atr=0.0)

## 2. Data

The chart frame gets a calendar warm-up; the extra timeframes get `TF_WARMUP_BARS` of **their own**
bars. `tf_frames` fetches each one through the normal incremental cache, so a repeat run is free.

In [ ]:
from quant.data import get_ohlcv, tf_frames

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
shown = (df.t >= pd.Timestamp(VIEW_START, tz="UTC")).sum()
print(f"chart {TF}: {len(df):,} bars   {df.t.min()}  ->  {df.t.max()}")
print(f"           {len(df)-shown:,} are WARM-UP; the chart opens on the {shown:,} from {VIEW_START}\n")

FRAMES = tf_frames(SYMBOL, PATTERN_TFS, view_start=VIEW_START, end=END,
                   warmup_bars=TF_WARMUP_BARS, source=SOURCE) if PATTERN_TFS else {}
for k, v in FRAMES.items():
    print(f"   {k:>4}: {len(v):>7,} bars from {v.t.min():%Y-%m-%d}")

## 3. What each timeframe found

Counted **before** plotting, and split by whether the shape confirms inside the window you are
looking at. A timeframe showing 0 in view is almost always the span arithmetic above, not a fault:
check `span_bars` against the number of bars your window holds at that timeframe.

In [ ]:
import numpy as np
from quant.indicators import chart_pattern_shapes_tuned

def in_view(frame, shapes):
    if not len(shapes):
        return 0
    t = pd.to_datetime(frame.t).dt.tz_convert("UTC").dt.tz_localize(None).to_numpy()
    v0 = int(np.searchsorted(t, np.datetime64(pd.Timestamp(VIEW_START))))
    return int((shapes.confirm_i >= v0).sum())

rows = []
for tf, frame in [(TF, df)] + list(FRAMES.items()):
    sh = chart_pattern_shapes_tuned(frame, patterns=PATTERNS)
    bars_in_view = (frame.t >= pd.Timestamp(VIEW_START, tz="UTC")).sum()
    rows.append(dict(tf=tf, bars=len(frame), bars_in_view=int(bars_in_view),
                     shapes=len(sh), in_view=in_view(frame, sh),
                     median_span=int(sh.span_bars.median()) if len(sh) else 0))
counts = pd.DataFrame(rows).set_index("tf")
counts["window_holds"] = (counts.bars_in_view / counts.median_span.clip(lower=1)).round(1)
print(counts.to_string())
print("\n`window_holds` = how many median-width shapes of that timeframe would even fit on screen."
      "\nBelow ~1 you should expect to see nothing, and widening the window is the fix.")

### The keys for `LEGEND_INSTANCES`

Copy the ones you want. A key with no `::` is the chart timeframe's own.

In [ ]:
keys = []
for tf, frame in [(None, df)] + list(FRAMES.items()):
    sh = chart_pattern_shapes_tuned(frame, patterns=PATTERNS)
    tt = pd.to_datetime(frame.t).dt.tz_convert("UTC").dt.tz_localize(None).to_numpy()
    v0 = int(np.searchsorted(tt, np.datetime64(pd.Timestamp(VIEW_START))))
    for name, n in sh[sh.confirm_i >= v0].pattern.value_counts().items():
        keys.append((name if tf is None else f"{tf}::{name}", int(n)))
    if SHOW_TRENDLINES:
        for side in ("support_trendline", "resistance_trendline"):
            keys.append((side if tf is None else f"{tf}::{side}", None))

print(f"{len(keys)} keys - put the ones you want in LEGEND_INSTANCES:")
for k, n in keys:
    print(f'    "{k}",' + (f"   # {n} in view" if n else ""))

## 4. The chart

**Three dropdowns under the title, each narrowing the next.**

| menu | picks | contains |
|---|---|---|
| 1 | **timeframe** | `▸ all timeframes`, then every timeframe that put something on the chart, coarsest first, with its pattern and occurrence counts |
| 2 | **pattern** | only that timeframe's families, with how many of each |
| 3 | **occurrence** | only that family's instances, earliest first |

Each menu is short *by construction*, which matters: plotly dropdowns do not scroll — they render
every row and clip whatever runs past the plot. A flat cross-timeframe list reached 51 rows, or
969px in a 900px figure, so its last entries could not be reached at all.

**Picking two of five, within one family — the legend.** Choosing a family in menu 2 promotes each
of its occurrences to its own legend entry (`Jun-06 20:00`, `May-31 09:40`, …). Click off the ones
you do not want. EMAs, S/R zones and other timeframes are untouched.

**Picking across families — `LEGEND_INSTANCES`.** The menus cannot express *one 15m resistance line
plus two 5m ones plus a few 1m shapes*, because a plotly button is stateless: each press replaces
the selection instead of adding to it, so choosing the 5m family drops your 15m pick. Naming those
families in `LEGEND_INSTANCES` puts **every one of their occurrences in the legend permanently**,
each starting off:

```
resistance trendline #4 · Jun-01 00:30 (broke)
5m · resistance trendline #1 · Jun-05 09:15 (broke)
5m · resistance trendline #2 · Jun-06 16:05 (faded)
1m · channel up #1 · Jun-09 03:35
```

Legend entries toggle independently and accumulate, and a dropdown press never removes one, so you
can click together any combination you like. Pin only the families you are working on — pinning
everything is a legend hundreds of rows long.

**Markers.** ★ = the bar a shape CONFIRMED (price closed through its boundary — often well to the
right of the shape itself). On an isolated trendline, ◆ = the bar the line became known and ✕ = the
bar it broke. A break needs the **close** through the line; a wick through does not count, and on
BTC 15m more than half of all pierces are wick-only.

**Higher-timeframe EMAs** are drawn as **steps**: a 1h EMA changes at each 1h close and holds flat
until the next one, because that is the only moment its value is final. A smooth interpolated line
across the hour would show numbers nobody could have read at the time. They get their own colours,
distinct from the chart's own EMAs, and their own legend toggles.

Hue is the **timeframe** and weight is the **period**, so `[("5m", 50, 100, 200), ("15m", 50, 100,
200)]` reads as two groups of three rather than six unrelated colours.

A long period needs a long history *in its own bars* — a 1h EMA200 needs 200 hours. Listing the
timeframe in `PATTERN_TFS` is what gives it that; otherwise the chart frame is resampled and a short
chart yields an all-NaN line, which you are warned about rather than left staring at nothing.

**The y-axis is scaled to the candles**, not to the geometry hanging off them. A 1h wedge that began
before the window can run from 86,000 to a 51,800 apex — the line is correct, but letting it set the
scale squeezed the price action into a third of the pane. Pass `y_from_price=False` for the old
behaviour.

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    df, patterns=PATTERNS, tuned=True, start_visible=START_VISIBLE,
    sr_zones=SHOW_SR_ZONES, zones_left=ZONE_LR, zones_right=ZONE_LR,
    cluster_pct=CLUSTER_PCT, min_touches=MIN_TOUCHES, extend=EXTEND,
    trendlines=SHOW_TRENDLINES, trendline_kwargs=TRENDLINE, tf_trendlines=TF_TRENDLINES,
    emas=EMAS, mtf_emas=MTF_EMAS, view_start=VIEW_START, line_picker=LINE_PICKER,
    zone_lookback_bars=ZONE_LOOKBACK,
    tf_frames=FRAMES, max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF,
    legend_instances=LEGEND_INSTANCES,
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}"
          + (f"   +  {', '.join(FRAMES)} patterns" if FRAMES else "")
          + "  ·  tuned settings",
    height=HEIGHT)

print(f"{len(fig.data)} traces, "
      f"{sum(len(m.buttons) for m in fig.layout.updatemenus)} dropdown entries, "
      f"{len(fig.to_json())/1e6:.2f} MB")
fig

### Open it in a browser — bigger, and it survives the notebook being closed

In [ ]:
from quant.viz import open_in_browser
open_in_browser(fig)

---

## 5. Which timeframes should I load?

The useful default is **one or two steps coarser than the chart**. That is where the extra
information is: a 4h support line drawn under a 15m chart is a level the 15m frame cannot see, and
it costs 0.15 s to compute.

Going *finer* is the opposite trade. A 1m layer under a 15m chart is 15× the bars, most of its
shapes are inside a single chart candle, and over nine days it found 155 of them. It is genuinely
useful for **one or two days**, and noise beyond that.

Run the cell below to see what a given set would cost before committing to it.

In [ ]:
import time

CANDIDATES = [("4h", "1h"), ("4h", "1h", "5m"), ("1h", "5m", "1m")]

for tfs in CANDIDATES:
    fr = tf_frames(SYMBOL, tfs, view_start=VIEW_START, end=END,
                   warmup_bars=TF_WARMUP_BARS, source=SOURCE)
    t0 = time.time()
    f = pattern_figure(df, tuned=True, trendlines=SHOW_TRENDLINES, view_start=VIEW_START,
                       emas=(), sr_zones=False, tf_frames=fr,
                       max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF)
    print(f"{str(tfs):<26} {time.time()-t0:5.1f}s   {len(f.data):>4} traces   "
          f"{len(f.to_json())/1e6:.2f} MB")

---

## 6. Config reference

### The multi-timeframe knobs

| knob | what it does |
|---|---|
| `tf_frames={"1h": df_1h}` | the extra layers. Load them with `quant.data.tf_frames` |
| `tf_patterns={"1h": [...]}` | a different pattern list per timeframe; omit to use `patterns` |
| `tf_trendlines` | `None` follows `trendlines`; `True`/`False` overrides it for the extras |
| `max_picker_instances` | occurrence budget for the extra timeframes, shared across a layer's families with a floor of one each — so it steers the total rather than capping it exactly. The chart's own is never trimmed |
| `max_shapes_per_tf` | occurrences drawn per pattern per timeframe, most recent kept |

Both caps bound **what is drawn and what is listed together**. Trimming only the menu is what used
to leave a legend reading `(11)` beside a dropdown offering nine, with two lines on the chart that
could not be isolated at all.
| `y_from_price` | scale the y-axis to the candles rather than to everything drawn |
| `mtf_emas` | higher-timeframe EMAs — `("1h", 200)`, `("1h", [50, 200])`, `("1h", 50, 100, 200)`, or a list of those. **Each entry needs periods**; a bare timeframe is an error, because that is `pattern_tfs`'s shape, not this one |
| `legend_instances` | families whose every occurrence becomes a permanent, independently toggleable legend entry — the only way to select across families. `"name"` (chart timeframe), `"5m::name"`, `"*::name"`, `"5m"`, or `"all"` |

### What the cascade costs

A dropdown button carries a `visible` vector **as long as the whole trace list**, so the menus cost
*buttons × traces* and a cascade nests: a timeframe button carries its family list, whose buttons
each carry an occurrence list. Naively that is unaffordable — four extra timeframes in one flat
arrangement measured **1,027 traces × 479 buttons ≈ 9.8 MB and an 11.4 s build**.

Three things keep it affordable. Each occurrence is listed **once**, under its own family, rather
than repeated in every wider selection. The wide `▸ all timeframes` state lists **no** families at
all — it asks you to pick one, which is both smaller and the only readable option at 51 rows. And
the caps above bound the occurrence count itself. Measured on BTC 15m with 1h, 5m and 1m loaded:
**461 traces, menus of 5 / 3 / 1 rows, 1.39 MB**, with the tallest menu you can ever open at 21
rows.

### The trendline settings (`TRENDLINE`)

| setting | meaning |
|---|---|
| `atr_mult` | how big a swing has to be to count as a pivot. Higher = fewer, more structural pivots |
| `min_touches` | pivots the line must pass through. **3 is the floor** — two points always fit a line exactly, so a two-touch "trendline" is not evidence of anything |
| `lookback_pivots` | how many recent pivots the fitter searches over |
| `line_tol_atr` | how far off the line a pivot may sit and still count as a touch |
| `min_span_bars` | the shortest line worth drawing |
| `max_age_bars` | a line is retired after this long without being touched |
| `break_buffer_atr` | how far past the line the **close** must go. `0.0` is a plain close through; `0.3` filters marginal breaks |

### Sister notebooks

* [`15_tuned_pattern_viewer`](15_tuned_pattern_viewer.ipynb) — single timeframe, all 22 patterns at
  their calibrated settings, plus the per-pattern settings table and the trendline tuning cells.
* [`14_pattern_viewer`](14_pattern_viewer.ipynb) — the same drawing with **one shared** detector
  configuration, for comparing patterns on equal terms.
* [`16_trendline_research`](16_trendline_research.ipynb) — the settings search that produced the
  numbers above.
